<a href="https://colab.research.google.com/github/dhiru69-tech/ML-Techniques-Lab-Project/blob/main/notebooks/day07_foundation_revision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 7: Foundation Revision
**Course:** BO CDA 205, Machine Learning Techniques
**Dataset:** German Credit (1000 customers, 21 columns)
**Syllabus link:** Foundation (data loading, EDA, encoding, scaling, splitting)

> **Goal of today:** Put the whole preprocessing workflow into reusable functions, with safety checks, so every later notebook can start from the same clean data.

## Foundation Recap (Day 1 to 6)

| Day | Topic | Key takeaway |
|---|---|---|
| 1 | Data loading | Use `shape`, `info`, `head` and `tail`. The dataset has 1000 rows and 21 columns, with no missing values |
| 2 | Understanding the data | Classes are imbalanced (70% good, 30% bad). `credit_amount` is right-skewed (mean 3271 vs median 2319.5) |
| 3 | EDA with graphs | Use histograms, boxplots, `groupby` and `crosstab(normalize='index')`. Compare percentages, not raw counts |
| 4 | Correlation | r ranges from -1 to +1. Check features against each other (redundancy) and against the target (usefulness) |
| 5 | Encoding | One-hot for nominal features, label encoding only for the target, ordinal encoding only when a real order exists (and we specify it) |
| 6 | Split and scaling | Split first, then fit on the training set only. On the test set, use `transform` only |

### Five rules to remember
1. **Split first**, then learn anything (scaler, encoder, feature selection) from the training set only
2. **`fit_transform` on train, `transform` on test** to avoid data leakage
3. **Accuracy alone is not enough.** The "always good" baseline scores 70% accuracy with 0 recall for bad customers
4. **No label encoding on nominal features**, because it creates a fake order
5. **Use `random_state=42` and `stratify=y`** so results are reproducible and class ratios are preserved

## 1. Preprocessing as Functions

A **function** is a named block of code we can call again and again. We write the workflow once and reuse it in every later notebook, so the split and preprocessing are always identical.

| Function | Job |
|---|---|
| `load_data()` | download the data, return features `X` and target `y` |
| `split_data()` | stratified 80/20 train and test split |
| `build_preprocessor()` | scale numeric columns, one-hot encode categorical columns |

In [1]:
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer


def load_data():
    """Load German Credit data. Returns X (features) and y (bad = 1, good = 0)."""
    data = fetch_openml(name="credit-g", version=1, as_frame=True)
    df = data.frame
    X = df.drop(columns=['class'])
    # astype(int) makes y plain 0/1 numbers instead of a category type
    y = df['class'].map({'good': 0, 'bad': 1}).astype(int)
    return X, y


def split_data(X, y, test_size=0.2, seed=42):
    """Stratified train-test split with a fixed seed so results are reproducible."""
    return train_test_split(X, y, test_size=test_size,
                            random_state=seed, stratify=y)


def build_preprocessor(X):
    """Scale numeric columns and one-hot encode categorical columns."""
    num_cols = X.select_dtypes(include='number').columns.tolist()
    cat_cols = X.select_dtypes(exclude='number').columns.tolist()

    pre = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), num_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ],
        sparse_threshold=0
    )
    pre.set_output(transform="pandas")   # keep column names in the output
    return pre

## 2. Running the Workflow

The order never changes: **load, split, fit the preprocessor on train only, transform both.**